# Main Wave Diffusion 16kHz training script
This is a script training the diffusion model on 16kHz samples and then umpsampling them to 32kHz using a second network.
### Imports

In [1]:
try: 
    import librosa
except: 
    !pip install librosa

#Set Dir 
import sys, os
sys.path.append(os.path.abspath('..'))

# Torch
import torch, torchaudio
from torch import nn, Tensor
import torch.optim as optim

# Utils
import numpy as np
import logging
import matplotlib.pyplot as plt


# Base Scripts
from Libraries.Utils import *
from Libraries.U_Net import *
from Libraries.Diffusion import *

## Setup

### Logging

In [2]:
logging_level: int = logging.INFO #LIGHT_DEBUG
logging.basicConfig(level=logging_level, format='%(asctime)s - %(levelname)s - %(message)s')
logger: logging.Logger = logging.getLogger(__name__)

### Initial Setup

In [3]:
remote_kernel: bool = True
device: str = "cuda" if torch.cuda.is_available() else "cpu"
training_data_name: str = "../Data/training_v2_full"
test_data_name: str = "../Data/unseen_test_data.npy"
model_name: str = "WaveDiffusion_16khz"
full_model_path: str = OS().path_to_remote_path("../Models/{}.pth".format(model_name), remote_kernel)

### Data Hyperparameters

In [4]:
n_samples: int = 2048
batch_size: int = 16
n_workers: int = 1

### Data loading
#### Training Data

In [6]:
md = ModelData()
md.load_data_from_path(data_path=OS().path_to_remote_path(training_data_name, remote_kernel))
md.create_validation_split(n_samples)
train_datatset, val_dataset = md.create_datasets()
train_dataloader, val_dataloader = md.create_dataloaders(batch_size, num_workers=n_workers)
logger.info(f"Created train dataset with length {len(md.train_dataset)} and validation dataset with length {len(md.val_dataset)}")

#### Test Data

In [5]:
md = ModelData()
md.load_data_from_path(data_path=OS().path_to_remote_path(test_data_name, remote_kernel))
md.create_validation_split()
test_dataset, _ = md.create_datasets()
test_dataloader, _ = md.create_dataloaders(batch_size, num_workers=n_workers)
logger.info(f"Created test dataset with length {len(md.train_dataset)}")

## Model & Training Setup
### Hyperparameters

In [6]:
b1, b2 = (0.9, 0.99)
lr: float = 2e-4
n_epochs: int = 100
start_epoch: int = 0
checkpoint_freq: int = 10
lr_restart_period: int = 30

### Training Settings

In [7]:
restart_training: bool = False
train_v_obj: bool = True
seq_length: int = 2**17
random_chunks: bool = True

### Model

In [8]:
u_net = UNet(in_channels=1, n_layers=4, base_channels=48, embed_dim=128, timesteps=1000, v_obj_sampler=True, kernel_size=11).to(device)

### Optimizers & Schedulers

In [10]:
optimizer = optim.AdamW(u_net.parameters(), lr, (b1, b2))
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=lr_restart_period, eta_min=1e-7, T_mult=2)

### Load Model

In [11]:
if os.path.exists(full_model_path):
    model = torch.load(full_model_path, map_location=device)
    u_net.load_state_dict(model["u_net"])
    if not restart_training:
        optimizer.load_state_dict(model["optim"])
        scheduler.load_state_dict(model["scheduler"])
        start_epoch = model.get("epoch", 0)
    logger.info(f"Model {model_name} loaded with {TrainingUtils().count_params(u_net)} Parameters")
else: 
    logger.info(f"Model {model_name} created with {TrainingUtils().count_params(u_net)} Parameters")

### Training Loss and Class setup

In [12]:
torch.backends.cudnn.benchmark = True
fb = Filterbank([0, 150, 1000, 4000, 8000], 512, 32000).to(device)
diffusion = Diffusion(noise_steps=1000, schedule="linear", inp_shape=[batch_size, 1, seq_length], device=device)

def loss_fn(pred_v: Tensor, real_v: Tensor, audio_input: Tensor, sigma_t: Tensor, diffusion: Diffusion, filter: Filterbank | PQMF, band_weights: list[float] = [0.1, 0.15, 0.2, 0.1]) -> Tensor:
    mse_loss = nn.MSELoss()(real_v, pred_v)
    a, b = diffusion.get_semicircle_weights(sigma_t)
    eps_from_pred_v = (pred_v + b * audio_input) / a
    x0_from_pred_v = (a * eps_from_pred_v - pred_v) / b
    pred_bands = filter.analysis(x0_from_pred_v)
    real_bands = filter.analysis(audio_input)
    band_weights = torch.tensor(band_weights, device=pred_v.device).view(1, -1, 1)
    filter_loss = (band_weights * torch.abs(pred_bands - real_bands)).mean()
    return mse_loss + filter_loss
    

## Training

### Training Loop

In [ ]:
logger.info(f"Training started on {device}")
loss_list: list = []
total_time: float = 0.0

for e in range(start_epoch, n_epochs + start_epoch):
    total_loss: float = 0
    total_reprod_quality: float = 0
    start_time: float = time.time()

    for b_idx, (audio, _) in enumerate(train_dataloader):
        audio: Tensor = audio.to(device)
        if audio.ndim == 2:
            audio = audio.unsqueeze(1)
        _, _, L = audio.shape
        if random_chunks:
            audio = TrainingUtils().random_crop_batch(audio, seq_length)
        else:
            audio = audio[..., :seq_length]
        audio = torchaudio.functional.resample(audio, orig_freq=32000, new_freq=16000)
        if not train_v_obj:
            x_t, noise, t = diffusion.prep_train_ddxm(audio)
            noise_recon = u_net(x_t, t)
            loss = loss_fn(noise, noise_recon)
            reprod_quality = TrainingUtils().reprod_quality_db(noise, noise_recon)
        else:
            true_vel, x_sigma_t, sigma_t  = diffusion.prep_train_v_obj(audio)
            pred_vel = u_net(x_sigma_t, sigma_t)
            loss = loss_fn(pred_vel, true_vel, audio, sigma_t, diffusion, fb)
            reprod_quality = TrainingUtils().reprod_quality_db(true_vel, pred_vel)

        if loss.isnan():
            break
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss
        total_reprod_quality += reprod_quality
    else:
        u_net.eval()
        total_val_loss: float = 0
        for audio,_ in val_dataloader:
            audio: Tensor = audio.to(device)
            if audio.ndim == 2:
                audio = audio.unsqueeze(1)
            _, _, L = audio.shape
            if random_chunks:
                audio = TrainingUtils().random_crop_batch(audio, seq_length)
            else:
                audio = audio[..., :seq_length]
            audio = torchaudio.functional.resample(audio, orig_freq=32000, new_freq=16000)
            with torch.no_grad():
                if not train_v_obj:
                    x_t, noise, t = diffusion.prep_train_ddxm(audio)
                    noise_recon = u_net(x_t, t)
                    total_val_loss += loss_fn(noise, noise_recon)
                else:
                    true_vel, x_sigma_t, sigma_t  = diffusion.prep_train_v_obj(audio)
                    pred_vel = u_net(x_sigma_t, sigma_t)
                    total_val_loss += loss_fn(pred_vel, true_vel, audio, sigma_t, diffusion, fb) 
        u_net.train()

        epoch_time = time.time() - start_time
        total_time += epoch_time
        remaining_time = int((total_time / (e + 1)) * (n_epochs - e - 1))
        avg_loss = total_loss / len(train_dataloader)
        avg_val_loss = total_val_loss / len(val_dataloader)
        avg_reprod_quality = total_reprod_quality / len(train_dataloader)
        loss_list.append({"avg_loss": avg_loss, "avg_val_loss": avg_val_loss, "avg_reprod_quality":avg_reprod_quality, "lr":optimizer.param_groups[0]['lr']})
        scheduler.step()

        if (e + 1) % 10 == 0:
            if not train_v_obj:
                out = diffusion.bwd_diffusion_ddim(u_net, [batch_size, 1, seq_length], n_steps=100, eta=0)
            else:
                out = diffusion.bwd_diffusion_v_obj(u_net, [batch_size, 1, seq_length], n_steps=100)
            ad = AudioData(out[0][0])
            ad.save_audio_file(f"Results/{model_name}_sample_e_{e+1}.wav")
            TrainingUtils().visualize_audio_and_spect(out[0][0])

        logger.info(f"Epoch {e + 1:03d}: Avg. Loss: {avg_loss:.3e} Avg. Val Loss: {avg_val_loss:.3e} Avg. reprod_quality {avg_reprod_quality:.3f} dB Remaining Time: {remaining_time // 3600:02d}h {(remaining_time % 3600) // 60:02d}min {round(remaining_time % 60):02d}s LR: {optimizer.param_groups[0]['lr']:.3e}")

        if checkpoint_freq > 0 and (e + 1) % checkpoint_freq == 0:
                checkpoint_path: str = f"{full_model_path[:-4]}_epoch_{e + 1:03d}.pth"
                torch.save({"u_net": u_net.state_dict(), "optim": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": e + 1}, checkpoint_path)
                if e + 1 != checkpoint_freq:
                    last_path: str = f"{full_model_path[:-4]}_epoch_{(e + 1) - checkpoint_freq:03d}.pth"
                    OS().del_if_exists(last_path)
                logger.light_debug(f"Checkpoint saved model to {checkpoint_path}")
        continue
    break
else:
    torch.save({"u_net": u_net.state_dict(), "optim": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": e + 1}, full_model_path)
    logger.light_debug(f"Saved model to {full_model_path}")

    if checkpoint_freq > 0:
        checkpoint_path: str = f"{full_model_path[:-4]}_epoch_{e + 1 - ((e + 1) % checkpoint_freq):03d}.pth"
        OS().del_if_exists(checkpoint_path)

### Visualize Training Run

In [ ]:
epochs = np.arange(1, len(loss_list) + 1)
avg_loss = [d['avg_loss'].detach().cpu().numpy() for d in loss_list]
avg_val_loss = [d['avg_val_loss'].cpu().numpy() for d in loss_list]
lr = [d['lr'] for d in loss_list]

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(epochs, avg_loss, label='Average Training Loss', color='blue', linewidth=2)
ax1.plot(epochs, avg_val_loss, label='Average Validation Loss', color='orange', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Metrics over Epochs')


ax2 = ax1.twinx()
ax2.plot(epochs, lr, label='Learning Rate', color='red', linestyle='--', linewidth=2)
ax2.set_ylabel('Learning Rate', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.legend(loc='upper right')
ax1.legend(loc='upper left')
ax1.grid(True)
plt.xlim(0,epochs[-1])

plt.tight_layout()
plt.show()

## Upsample Module

In [13]:
class ResBlockNoEmbed(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3, dilation: int = 1) -> None:
        super().__init__()
        pad = (kernel_size - 1) // 2 * dilation

        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.norm1 = nn.BatchNorm1d(channels)
        self.norm2 = nn.BatchNorm1d(channels)

        self.act = nn.SiLU()
    
    def forward(self, x: Tensor) -> Tensor:
        out = self.conv1(self.act(self.norm1(x)))
        out = self.conv2(self.act(self.norm2(out)))
        return self.act(out + x)

class Upsample(nn.Module):
    def __init__(self, in_channels: int, base_channels: int, kernel_size: int, in_res_blocks: int, out_res_bloks: int) -> None:
        super(Upsample, self).__init__()
        padding = (kernel_size - 1) // 2
        out_pad = 2 + 2 * padding - kernel_size
        in_conv = nn.Conv1d(in_channels, base_channels, kernel_size=kernel_size, stride=1, padding=padding)
        out_conv = nn.Conv1d(base_channels, in_channels, kernel_size=kernel_size, stride=1, padding=padding)
        in_blocks = [ResBlockNoEmbed(base_channels, kernel_size, dilation=2**i) for i in range(in_res_blocks)]
        out_blocks = [ResBlockNoEmbed(base_channels, kernel_size, dilation=2**(out_res_bloks - 1 - i)) for i in range(out_res_bloks)]
        upsample = nn.ConvTranspose1d(base_channels, base_channels, kernel_size=kernel_size, stride=2, padding=padding, output_padding=out_pad)
        self.model = nn.Sequential(
            in_conv,
            *in_blocks,
            upsample,
            *out_blocks,
            out_conv
        )
    def forward(self, x: Tensor) -> Tensor:
        return self.model(x)

In [14]:
ups_model = Upsample(in_channels=1, base_channels=64, kernel_size=11, in_res_blocks=4, out_res_bloks=4).to(device)

### Training Settings

In [15]:
model_name_upsampler = model_name + "_upsampler"
full_model_path_upsampler = OS().path_to_remote_path("../Models/{}.pth".format(model_name_upsampler), remote_kernel)
restart_training: bool = False
train_v_obj: bool = True
seq_length: int = 2**17
random_chunks: bool = True


b1, b2 = (0.9, 0.99)
lr: float = 2e-4
n_epochs: int = 100
start_epoch: int = 0
checkpoint_freq: int = 10
lr_restart_period: int = 30

### Optimizers & Schedulers

In [16]:
optimizer = optim.AdamW(ups_model.parameters(), lr, (b1, b2))
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=lr_restart_period, eta_min=1e-7, T_mult=2)

### Load Model

In [18]:
if os.path.exists(full_model_path_upsampler):
    model = torch.load(full_model_path_upsampler, map_location=device)
    ups_model.load_state_dict(model["ups_model"])
    if not restart_training:
        optimizer.load_state_dict(model["optim"])
        scheduler.load_state_dict(model["scheduler"])
        start_epoch = model.get("epoch", 0)
    logger.info(f"Model {model_name_upsampler} loaded with {TrainingUtils().count_params(ups_model)} Parameters")
else: 
    logger.info(f"Model {model_name_upsampler} created with {TrainingUtils().count_params(ups_model)} Parameters")

In [13]:
logger.info(f"Training started on {device}")
loss_list: list = []
total_time: float = 0.0

for e in range(start_epoch, n_epochs + start_epoch):
    total_loss: float = 0
    start_time: float = time.time()

    for b_idx, (audio, _) in enumerate(train_dataloader):
        audio: Tensor = audio.to(device)
        if audio.ndim == 2:
            audio = audio.unsqueeze(1)
        _, _, L = audio.shape
        if random_chunks:
            audio = TrainingUtils().random_crop_batch(audio, seq_length)
        else:
            audio = audio[..., :seq_length]
        audio_in = torchaudio.functional.resample(audio, orig_freq=32000, new_freq=16000)
        out = ups_model(audio_in)
        loss = torch.nn.MSELoss()(out, audio)
        

        if loss.isnan():
            break
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss
    else:
        ups_model.eval()
        total_val_loss: float = 0
        for audio,_ in val_dataloader:
            audio: Tensor = audio.to(device)
            if audio.ndim == 2:
                audio = audio.unsqueeze(1)
            _, _, L = audio.shape
            if random_chunks:
                audio = TrainingUtils().random_crop_batch(audio, seq_length)
            else:
                audio = audio[..., :seq_length]
            audio_in = torchaudio.functional.resample(audio, orig_freq=32000, new_freq=16000)
            with torch.no_grad():
                out = ups_model(audio_in)
            total_val_loss += torch.nn.MSELoss()(out, audio)
        ups_model.train()

        epoch_time = time.time() - start_time
        total_time += epoch_time
        remaining_time = int((total_time / (e + 1)) * (n_epochs - e - 1))
        avg_loss = total_loss / len(train_dataloader)
        avg_val_loss = total_val_loss / len(val_dataloader)
        loss_list.append({"avg_loss": avg_loss, "avg_val_loss": avg_val_loss, "lr":optimizer.param_groups[0]['lr']})
        scheduler.step()

        logger.info(f"Epoch {e + 1:03d}: Avg. Loss: {avg_loss:.3e} Avg. Val Loss: {avg_val_loss:.3e} Remaining Time: {remaining_time // 3600:02d}h {(remaining_time % 3600) // 60:02d}min {round(remaining_time % 60):02d}s LR: {optimizer.param_groups[0]['lr']:.3e}")

        if checkpoint_freq > 0 and (e + 1) % checkpoint_freq == 0:
                checkpoint_path: str = f"{full_model_path_upsampler[:-4]}_epoch_{e + 1:03d}.pth"
                torch.save({"ups_model": ups_model.state_dict(), "optim": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": e + 1}, checkpoint_path)
                if e + 1 != checkpoint_freq:
                    last_path: str = f"{full_model_path_upsampler[:-4]}_epoch_{(e + 1) - checkpoint_freq:03d}.pth"
                    OS().del_if_exists(last_path)
                logger.light_debug(f"Checkpoint saved model to {checkpoint_path}")
        continue
    break
else:
    torch.save({"ups_model": ups_model.state_dict(), "optim": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": e + 1}, full_model_path_upsampler)
    logger.light_debug(f"Saved model to {full_model_path_upsampler}")

    if checkpoint_freq > 0:
        checkpoint_path: str = f"{full_model_path_upsampler[:-4]}_epoch_{e + 1 - ((e + 1) % checkpoint_freq):03d}.pth"
        OS().del_if_exists(checkpoint_path)

In [ ]:
epochs = np.arange(1, len(loss_list) + 1)
avg_loss = [d['avg_loss'].detach().cpu().numpy() for d in loss_list]
avg_val_loss = [d['avg_val_loss'].cpu() for d in loss_list]
lr = [d['lr'] for d in loss_list]

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(epochs, avg_loss, label='Average Training Loss', color='blue', linewidth=2)
ax1.plot(epochs, avg_val_loss, label='Average Validation Loss', color='orange', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Metrics over Epochs')
ax1.legend(loc='upper left')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(epochs, lr, label='Learning Rate', color='red', linestyle='--', linewidth=2)
ax2.set_ylabel('Learning Rate', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

## Inference

In [22]:
idx = 1
n_steps = 100
if not train_v_obj:
    out = diffusion.bwd_diffusion_ddim(u_net, [batch_size, 1, seq_length//2], n_steps=n_steps, eta=0)
else:
    out = diffusion.bwd_diffusion_v_obj(u_net, [batch_size // 2, 1, seq_length//2], n_steps=n_steps)

out = np.clip(out, -1, 1)
ad = AudioData(out[idx][0], sr=16000)
ad.save_audio_file(f"Results/{model_name}_test_sample{idx}_s{n_steps}.wav")
TrainingUtils().visualize_audio_and_spect(out[idx][0])
with torch.no_grad():
    upsampled_out = ups_model(torch.tensor(out).to(device)).cpu().numpy()
ad = AudioData(upsampled_out[idx][0])
ad.save_audio_file(f"Results/{model_name}_test_sample{idx}_upsampled.wav")
TrainingUtils().visualize_audio_and_spect(upsampled_out[idx][0])
